# 04 — Validation evaluation

This is the main Referee evaluation notebook. It runs the `recognition.evaluate` runner in the Python 3.13 model environment against the **validation** split. The held-out test split requires a freeze record and is not for tuning.

The runner reports encoded-proxy AP and known/reviewed-label AP separately. The checkpoint filename score is not reviewed-label accuracy.


## Protocol

- `foreground` is the shared Guesser input contract.
- `rgb_resize` matches the current `training/dataset.py` transform.
- `auto` probes both on at most 200 rows and selects only when the winner is close to the checkpoint score and clearly separated. Otherwise it stops and asks for provenance.
- Validation metrics are for threshold review; do not tune on test.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from referee_evaluation import notebook_tools

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'recognition').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'


## Options

Execution is opt-in. Use `LIMIT = 200` for a wiring run, then `LIMIT = None` for full validation after checking runtime and data coverage.


In [ ]:
RUN_EVALUATION = False
PROFILE = 'sample'          # 'sample' or 'train'
SPLIT = 'validation'
CHECKPOINT = MODEL_PROJECT / 'checkpoints' / 'referee-epoch=04-val_mean_ap=0.5564.ckpt'
PREPARED_DIR = MODEL_PROJECT / 'data' / '.local' / 'data' / PROFILE / 'prepared'
OUTPUT_ROOT = MODEL_PROJECT / 'checkpoints' / 'referee-evaluation' / PROFILE / 'validation-v1'
PREPROCESSING = 'auto'      # auto, foreground, rgb_resize
BATCH_SIZE = 32
BOOTSTRAP = 1000
LIMIT = 200                # None for full validation
OVERWRITE = False
TARGET_SCORE = None        # None uses the checkpoint filename score

if SPLIT != 'validation':
    raise ValueError('This notebook is validation-only.')
if LIMIT is not None and LIMIT < 1:
    raise ValueError('LIMIT must be positive or None.')
if PREPROCESSING not in {'auto', 'foreground', 'rgb_resize'}:
    raise ValueError('Unknown preprocessing.')

print('Evaluation enabled:', RUN_EVALUATION)
print('Checkpoint:', CHECKPOINT)
print('Prepared:', PREPARED_DIR)
print('Output:', OUTPUT_ROOT)


## Readiness

If prepared data is missing, run notebook 01 for the same profile and export the trainer split. The evaluator never downloads data.


In [ ]:
manifest = PREPARED_DIR / 'manifests' / f'{SPLIT}.jsonl'
READY = (
    CHECKPOINT.is_file()
    and manifest.is_file()
    and (MODEL_PROJECT / 'recognition' / 'evaluate.py').is_file()
)
print('Checkpoint present:', CHECKPOINT.is_file())
print('Validation manifest present:', manifest.is_file())
print('Evaluator present:', (MODEL_PROJECT / 'recognition' / 'evaluate.py').is_file())
if not READY:
    print('Not ready. Prepare the profile with notebook 01, then rerun this check.')


## Run the evaluator

The command is disabled by default. A nonzero exit or a user interrupt raises here; a successful run writes the ignored output directory.


In [ ]:
MODEL_ENV = dict(os.environ)
MODEL_ENV.pop('PYTHONPATH', None)
MODEL_ENV.pop('PYTHONHOME', None)

if not RUN_EVALUATION:
    print('Evaluation disabled. Review the options, then set RUN_EVALUATION = True.')
elif not READY:
    print('Evaluation skipped: data/checkpoint prerequisites are missing.')
else:
    command = [
        'uv', 'run', '--project', str(MODEL_PROJECT), '--frozen', '--no-sync',
        'python', '-m', 'recognition.evaluate',
        '--checkpoint', str(CHECKPOINT),
        '--prepared-dir', str(PREPARED_DIR),
        '--split', SPLIT,
        '--output-dir', str(OUTPUT_ROOT),
        '--preprocessing', PREPROCESSING,
        '--batch-size', str(BATCH_SIZE),
        '--bootstrap', str(BOOTSTRAP),
    ]
    if LIMIT is not None:
        command.extend(['--limit', str(LIMIT)])
    if TARGET_SCORE is not None:
        command.extend(['--target-score', str(TARGET_SCORE)])
    if OVERWRITE:
        command.append('--overwrite')
    print('Launching:', ' '.join(command))
    if not notebook_tools.run_interruptible_command(command, cwd=MODEL_PROJECT, env=MODEL_ENV):
        raise RuntimeError('Evaluation interrupted.')


## Summary

Read the emitted `summary.json`. Encoded AP is comparable to the checkpoint training metric; known/reviewed AP is the game-facing slice.


In [ ]:
SUMMARY_PATH = OUTPUT_ROOT / 'summary.json'
if not SUMMARY_PATH.is_file():
    print('No summary yet. Run the evaluator or check OUTPUT_ROOT.')
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    rows = []
    for name in ('text', 'number', 'symbol', 'circle'):
        channel = summary['channels'][name]
        rows.append({
            'channel': name,
            'encoded_ap': channel['encodedAp'],
            'known_ap': channel.get('knownAp'),
            'reviewed_ap': channel.get('reviewedAp'),
            'proxy_ap': channel.get('proxyAp'),
            'known': channel['support'].get('known'),
            'positive': channel['support'].get('positive'),
        })
    display(pd.DataFrame(rows).set_index('channel'))
    display(pd.DataFrame([summary['coverage']]).T)
    print('Preprocessing:', summary['preprocessing'])
    print('Mean encoded AP (text+number):', summary['meanEncodedApTextNumber'])
    print('Probe:', summary.get('probe'))


## Threshold trade-off

The curves below are validation slices for policy review, not a frozen game threshold.


In [ ]:
if SUMMARY_PATH.is_file():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for axis, name in zip(axes, ('text', 'number')):
        points = summary['channels'][name].get('thresholds') or []
        if points:
            axis.plot([p['threshold'] for p in points], [p['precision'] for p in points], label='precision')
            axis.plot([p['threshold'] for p in points], [p['recall'] for p in points], label='recall')
        axis.set_title(name)
        axis.set_xlabel('score threshold')
        axis.set_ylim(0, 1)
        axis.grid(True, alpha=0.3)
        axis.legend()
    axes[0].set_ylabel('score')
    plt.tight_layout()
    plt.show()
else:
    print('No summary yet; run the evaluator first.')


## Artifacts and limits

- Outputs: `predictions.jsonl`, `summary.json`, `report.md`, optional `preprocessing-probe.json` under ignored `models/referee/checkpoints/referee-evaluation/`.
- These are validation metrics, not gameplay accuracy or model-performance claims.
- The held-out test split is separate; do not tune thresholds on it.
- No replay/persistence timing is measured here.
